# Feature engineering: store the market state

Each tick's market profile goes into Feast — offline for the training join,
online (DynamoDB via moto) so the Flink job can enrich live events.

## The streaming stack

In [ ]:
%%bash
cat docker-compose.yml

In [ ]:
%%bash
make up-moto up-kafka

In [ ]:
%%bash
make ps

## Build the tables + the replay stream

In [ ]:
%%bash
python -m electricity_price_streaming.prepare_data

In [ ]:
%%bash
mkdir -p feature_repo/data && cp data/processed/market_features.parquet data/processed/direction_labels.parquet feature_repo/data/

In [ ]:
%%bash
cd feature_repo && feast apply

In [ ]:
%%bash
aws dynamodb list-tables --endpoint-url http://electricity-price-streaming-moto:5000

## Materialize the market profile online

In [ ]:
%%bash
cd feature_repo && feast materialize 1996-01-01T00:00:00 2030-01-01T00:00:00

In [ ]:
from feast import FeatureStore

store = FeatureStore(repo_path="feature_repo")
store.get_online_features(
    features=["market_profile:nswprice", "market_profile:nswdemand"],
    entity_rows=[{"market_tick_id": "tick-00000"}],
).to_dict()

## Training frame (point-in-time)

In [ ]:
import pandas as pd
from electricity_price_streaming.feature_contract import (
    FEATURE_COLUMNS,
    LABEL_COLUMN,
)

entities = pd.read_parquet("data/processed/direction_labels.parquet")
training = store.get_historical_features(
    entity_df=entities,
    features=[f"market_profile:{c}" for c in FEATURE_COLUMNS]
    + [f"direction_labels:{LABEL_COLUMN}"],
).to_df()
training.head()